# Fase 1 · Diseño del centro de datos e ingesta  (RA1 · UD1)

**Reto:** la autoridad portuaria quiere una plataforma que reúna los datos del puerto (escalas de buques,
posiciones AIS, sensores, contenedores) y responda preguntas de negocio. Vosotros diseñáis el almacenamiento,
ingestáis los datos, elegís formatos, los procesáis y presentáis los resultados al «cliente».

| Parte | Qué hacéis | CE |
|---|---|---|
| A | Explorar los datos crudos en HDFS | 1.a, 1.b |
| B | Leer con esquema explícito y controlar registros corruptos | 1.b |
| C | Comparar formatos de almacenamiento | 1.c |
| D | Zona *bronze* en Parquet particionado | 1.c, 1.d |
| E | Procesar con Spark SQL | 1.d |
| F | Presentar resultados al cliente | 1.e |

Las celdas marcadas con **✏️ TAREA** son vuestras. Todo lo demás funciona tal cual: ejecutadlo, leedlo y entendedlo
(os puedo pedir que expliquéis cualquier línea).

Zonas del lago de datos del proyecto:
```
/puerto/raw        tal como llegan (CSV, JSON...)             ← ingesta.sh
/puerto/bronze     mismos datos, tipados, en formato columnar  ← esta fase
/puerto/silver     datos validados y limpios                   ← fase 3
/puerto/gold       agregados listos para negocio               ← fases 1 y 5
/puerto/cuarentena filas que no superan la calidad             ← fase 3
/puerto/control    manifiestos, registros de ingesta e informes
```

In [ ]:
from pyspark.sql import functions as F, types as T
from puerto.sesion import crear_spark, cronometro, sello
PAREJA = "00"   # ✏️ vuestro número de pareja: su salida debe verse en las capturas de la memoria
sello(PAREJA)
from puerto import hdfs_web as h
import pandas as pd, matplotlib.pyplot as plt, os
spark = crear_spark("F1-ingesta-formatos")
os.makedirs("/workspace/entregas/F1", exist_ok=True)

## A · Exploración de los datos crudos
La ingesta por lotes (`.\bda.ps1 ingesta`) ya ha copiado los ficheros a `/puerto/raw`. Revisad el script
`scripts/hdfs/ingesta.sh`: crea las zonas, copia con `put -f` (idempotente) y deja un registro en `/puerto/control`.

In [ ]:
for carpeta in ["maestros", "escalas", "negocio", "contenedores", "sensores", "ais"]:
    r = h.resumen(f"/puerto/raw/{carpeta}")
    print(f"{carpeta:13} {r['ficheros']:4} ficheros  {r['tamano']:>10} lógicos  {r['en_disco']:>10} en disco")

**✏️ TAREA A1.** ¿Por qué el espacio «en disco» es aproximadamente el triple del lógico? ¿Qué propiedad de
`docker/hadoop.env` lo explica? Contestad en la memoria.

**✏️ TAREA A2.** La ingesta actual es **por lotes** (todo de golpe). Los datos AIS en realidad llegan de forma
continua. Describid en la memoria cómo haríais una ingesta **incremental** (idea: `scripts/hdfs/simular_llegada_ais.sh`)
y qué herramientas se usan en la industria (Kafka, NiFi, Flume, Spark Structured Streaming...).
Indicad para cada fuente del proyecto: frecuencia de llegada, volumen/día y mecanismo de ingesta propuesto.

## B · Lectura con esquema explícito
Inferir el esquema (`inferSchema=True`) obliga a leer los datos dos veces y puede equivocarse.
En producción **siempre** se declara el esquema. Además, pedimos a Spark que no descarte en silencio
las líneas rotas, sino que las guarde en una columna `_corrupto`.

In [ ]:
esquema_ais = T.StructType([
    T.StructField("mmsi", T.StringType()),
    T.StructField("ts", T.StringType()),
    T.StructField("lat", T.DoubleType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("sog", T.DoubleType()),
    T.StructField("cog", T.DoubleType()),
    T.StructField("estado_nav", T.IntegerType()),
    T.StructField("estacion", T.StringType()),
    T.StructField("_corrupto", T.StringType()),
])
ais_raw = (spark.read.schema(esquema_ais)
           .option("mode", "PERMISSIVE")
           .option("columnNameOfCorruptRecord", "_corrupto")
           .json("/puerto/raw/ais/"))
ais_raw.cache()
print("Posiciones AIS:", ais_raw.count())
print("Líneas corruptas:", ais_raw.filter(F.col("_corrupto").isNotNull()).count())
ais_raw.filter(F.col("_corrupto").isNotNull()).select("_corrupto").show(3, truncate=80)

In [ ]:
esquema_escalas = T.StructType([
    T.StructField(c, t) for c, t in [
        ("id_escala", T.StringType()), ("imo", T.StringType()), ("buque", T.StringType()),
        ("id_naviera", T.StringType()), ("id_muelle", T.StringType()), ("tipo_trafico", T.StringType()),
        ("operacion", T.StringType()), ("mercancia", T.StringType()), ("toneladas", T.DoubleType()),
        ("teu", T.IntegerType()), ("eta", T.StringType()), ("ata", T.StringType()), ("atd", T.StringType()),
        ("origen", T.StringType()), ("destino", T.StringType()), ("practicaje", T.StringType()),
        ("pasajeros", T.IntegerType())]
])
escalas_raw = spark.read.schema(esquema_escalas).option("header", True).csv("/puerto/raw/escalas/")
escalas_raw.printSchema()
escalas_raw.show(5, truncate=False)

**✏️ TAREA B1.** Leed `/puerto/raw/negocio/facturas_2025.csv`. Fijaos: separador `;`, **coma decimal** y fechas
`dd/mm/aaaa` (típico de un ERP español). Leedlo con esquema de texto y convertid `cantidad`, `precio_unitario` e
`importe` a número (pista: `F.regexp_replace(col, ",", ".").cast("double")`). Mostrad 5 filas y el esquema.

In [ ]:
# ✏️ TAREA B1

## C · ¿Qué formato de almacenamiento elegimos?
Escribimos las mismas posiciones AIS en seis formatos y medimos: **tiempo de escritura**, **tamaño en HDFS** y
**tiempo de dos consultas típicas**. La celda tarda unos minutos.

In [ ]:
ais = ais_raw.filter(F.col("_corrupto").isNull()).drop("_corrupto")
formatos = [
    ("csv",            "csv",     {"header": "true"}),
    ("json",           "json",    {}),
    ("avro",           "avro",    {}),
    ("orc",            "orc",     {}),
    ("parquet_snappy", "parquet", {"compression": "snappy"}),
    ("parquet_zstd",   "parquet", {"compression": "zstd"}),
]
resultados = []
for nombre, fmt, opciones in formatos:
    ruta = f"/puerto/bench/ais_{nombre}"
    t = {}
    with cronometro(f"escritura {nombre}", t):
        ais.write.mode("overwrite").format(fmt).options(**opciones).save(ruta)
    lector = spark.read.format(fmt).options(**opciones)
    if fmt in ("csv", "json"):
        lector = lector.schema(ais.schema)
    df = lector.load(ruta)
    with cronometro(f"consulta completa {nombre}", t):
        df.groupBy("mmsi").agg(F.avg("sog")).count()
    with cronometro(f"consulta 2 columnas {nombre}", t):
        df.select("lat", "lon").where("lat > 37.1").count()
    r = h.resumen(ruta)
    resultados.append({"formato": nombre, "MB": round(r["bytes"] / 2**20, 1),
                       "escritura_s": t[f"escritura {nombre}"],
                       "consulta_completa_s": t[f"consulta completa {nombre}"],
                       "consulta_2col_s": t[f"consulta 2 columnas {nombre}"]})
comparativa = pd.DataFrame(resultados).set_index("formato")
comparativa.to_csv("/workspace/entregas/F1/comparativa_formatos.csv")
comparativa

In [ ]:
ax = comparativa[["MB"]].plot.bar(figsize=(8, 3.5), legend=False, title="Tamaño en HDFS (MB, sin réplicas)")
plt.tight_layout(); plt.savefig("/workspace/entregas/F1/formatos_tamano.png", dpi=120); plt.show()
ax = comparativa[["escritura_s", "consulta_completa_s", "consulta_2col_s"]].plot.bar(figsize=(8, 3.5), title="Tiempos (s)")
plt.tight_layout(); plt.savefig("/workspace/entregas/F1/formatos_tiempos.png", dpi=120); plt.show()

**✏️ TAREA C1.** Repetid la medición una segunda vez: ¿cambian los tiempos? ¿Por qué la primera ejecución suele ser
más lenta? **✏️ TAREA C2.** Explicad en la memoria, con vuestros números: formato por filas vs. columnar, compresión,
esquema embebido, lectura de solo 2 columnas (*column pruning*) y si el formato es «divisible» (*splittable*).
Elegid y justificad el formato para las zonas bronze, silver y gold.

## D · Zona *bronze*: Parquet particionado por fecha
Añadimos **metadatos de linaje** (fichero de origen y momento de ingesta) y particionamos por fecha:
cada día queda en su carpeta `fecha=AAAA-MM-DD`, y las consultas que filtran por fecha solo leen esas carpetas
(*partition pruning*).

In [ ]:
ais_bronze = (ais
    .withColumn("ts", F.to_timestamp("ts"))
    .withColumn("fecha", F.to_date("ts"))
    .withColumn("_fichero_origen", F.input_file_name())
    .withColumn("_ingestado", F.current_timestamp()))
(ais_bronze.repartition("fecha")
    .write.mode("overwrite").partitionBy("fecha").parquet("/puerto/bronze/ais"))
h.ls("/puerto/bronze/ais")

In [ ]:
b = spark.read.parquet("/puerto/bronze/ais")
consulta = b.filter(F.col("fecha") == "2025-06-15").groupBy("estado_nav").count()
consulta.explain()      # buscad «PartitionFilters» en el plan físico
consulta.show()

Hacemos lo mismo con las **escalas** (particionadas por año). Ojo: algunas fechas vienen como `dd/MM/yyyy HH:mm`,
por eso probamos los dos formatos con `coalesce`.

In [ ]:
def a_timestamp(c):
    return F.coalesce(F.to_timestamp(c, "yyyy-MM-dd HH:mm:ss"), F.to_timestamp(c, "dd/MM/yyyy HH:mm"))

escalas_bronze = (escalas_raw
    .withColumn("eta", a_timestamp("eta")).withColumn("ata", a_timestamp("ata")).withColumn("atd", a_timestamp("atd"))
    .withColumn("anio", F.year("eta"))
    .withColumn("_fichero_origen", F.input_file_name())
    .withColumn("_ingestado", F.current_timestamp()))
escalas_bronze.write.mode("overwrite").partitionBy("anio").parquet("/puerto/bronze/escalas")
h.ls("/puerto/bronze/escalas")

**✏️ TAREA D1.** Cread la zona bronze de **sensores** (`/puerto/bronze/sensores`, particionado por fecha; el campo `ts`
trae zona horaria `+02:00`), de **contenedores** (`/puerto/bronze/contenedores`) y de los tres **maestros**
(`/puerto/bronze/maestros/<nombre>`). Declarad siempre el esquema.
**✏️ TAREA D2.** Explicad el problema de los «ficheros pequeños» (*small files*) en HDFS y por qué hacemos `repartition("fecha")`
antes de escribir.

In [ ]:
# ✏️ TAREA D1

## E · Procesamiento con Spark SQL
Registramos las tablas bronze como vistas y respondemos preguntas del cliente. Filtramos de momento los valores
absurdos de toneladas; la limpieza completa llega en la fase 3.

In [ ]:
spark.read.parquet("/puerto/bronze/escalas").createOrReplaceTempView("escalas")
spark.read.option("header", True).csv("/puerto/raw/maestros/navieras.csv").createOrReplaceTempView("navieras")

trafico = spark.sql('''
    SELECT date_format(eta, 'yyyy-MM') AS mes, tipo_trafico,
           COUNT(*) AS escalas, ROUND(SUM(toneladas) / 1000, 1) AS miles_t
    FROM escalas
    WHERE toneladas BETWEEN 0 AND 500000
    GROUP BY 1, 2 ORDER BY 1, 2''')
trafico.write.mode("overwrite").parquet("/puerto/gold/f1_trafico_mensual")
trafico.show(10)

In [ ]:
espera = spark.sql('''
    SELECT tipo_trafico, year(eta) AS anio, quarter(eta) AS trimestre,
           ROUND(AVG((unix_timestamp(ata) - unix_timestamp(eta)) / 3600), 1) AS espera_media_h,
           COUNT(*) AS escalas
    FROM escalas
    WHERE ata IS NOT NULL AND eta IS NOT NULL
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3''')
espera.write.mode("overwrite").parquet("/puerto/gold/f1_espera_trimestral")
espera.filter("tipo_trafico = 'contenedores'").show()

**✏️ TAREA E1.** Escribid y guardad en `/puerto/gold/` al menos **tres** consultas más que respondan a:
1. Las 10 navieras con más toneladas movidas en 2025 (con su nombre, uniendo con `navieras`).
2. La ocupación de cada muelle: horas atracadas (`atd - ata`) por mes.
3. Una pregunta que propongáis vosotros usando AIS o sensores (p. ej. velocidad media de entrada, horas con viento > 14 m/s).

In [ ]:
# ✏️ TAREA E1

## F · Presentación al cliente
El cliente no lee tablas de Spark. Convertimos los resultados *gold* (pequeños) a pandas y hacemos gráficos claros.

In [ ]:
pdf = espera.filter("tipo_trafico = 'contenedores'").toPandas()
pdf["periodo"] = pdf["anio"].astype(str) + "-T" + pdf["trimestre"].astype(str)
ax = pdf.plot(x="periodo", y="espera_media_h", marker="o", legend=False, figsize=(8, 3.5),
              title="Espera media en fondeo · portacontenedores (horas)")
ax.set_xlabel(""); ax.set_ylabel("horas"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/workspace/entregas/F1/espera_contenedores.png", dpi=120); plt.show()

**✏️ TAREA F1.** Preparad un **informe ejecutivo de una página** en `entregas/F1/MEMORIA_F1.md` (apartado CE1.e) para la
dirección del puerto: 3 gráficos como máximo, 3 conclusiones en lenguaje no técnico y una recomendación.
¿Qué ha pasado con la espera de los portacontenedores en 2025?

In [ ]:
spark.stop()